# Phase 3: 02 B1 LoRA — own model, one T4

This is a separate, self-contained notebook. Use **Runtime → Change runtime type → T4 GPU** in Colab. Kaggle users must enable GPU and Internet and persist session output using Save Version.

No pretrained model is downloaded: only your archived Phase1 model and official RR data are used. Adapters are implemented from scratch. FP16 is used on T4. Scores and GPU execution are **not verified in the authoring environment**.

Four notebooks cover four model families, LoRMA/LoRA/IA3 and mandatory distillation. Four alone do **not** satisfy the document's opening six-submission minimum. Record the real hardware and LLM assistance honestly.

## Files to upload to your own Drive
Required: **Phase3_B1_Input.zip**. Upload via Drive, not a giant browser `files.upload()` call. Put them in `MyDrive/CS781_Phase3/inputs/`.

Optional official dataset ZIP: put **Phase_3_RR.zip** in the same folder. Otherwise log into Hugging Face with an account already granted access and let the notebook fetch the seven official files. Never paste credentials into a shared notebook.

Persistent output: `MyDrive/CS781_Phase3/runs/02_B1_LoRA/`. Do not change settings while resuming the same run. Start a new run name for new hyperparameters.


In [ ]:
# Keep Colab's installed CUDA-enabled torch. No PEFT or transformers models.
%pip -q install "tokenizers>=0.20,<0.23" "huggingface_hub>=0.26,<2" "scikit-learn>=1.4,<2"
import sys,torch,platform,shutil
print(platform.python_version(),torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0))
print('Free local GB:',round(shutil.disk_usage('/').free/1e9,2))


In [ ]:
# Embedded source; no separate source upload required.
import base64,hashlib,io,zipfile,pathlib,sys
source_bytes=base64.b64decode('UEsDBBQAAAAIACmrUF2FLY1VRQAAAEkAAAAVAAAAcGhhc2UzX3JyL19faW5pdF9fLnB5BcGxDYBACAXQ3il+aKy0cRFXIB7KJXgYwLi+7xHRrpyCDaFSHv1gW8JNwI2f4uo+4CdKBW9KzAn/BkwuNtzexHIloukHUEsDBBQAAAAIACisUF1GECGF1AIAAM0FAAAbAAAAcGhhc2UzX3JyL2FyY2hpdmVfaW5wdXRzLnB5dVRNb9xGDL3rV9B7iKREmKyNNi02UIAgSFAfGhiw00OKQpiVqBVjzQdmqHi7v77gSOt0UfUizHD48R75qM1mc2/0OMLkR6c70KEd6DtGaJ1lTZbsAfCoWwYX6EBWj9AO2D56R5bBoNljiBVYxxCQgyaLHTwhHQaOarPZZGS8CwzforNVHCamsTqR72nErA/OgNc8jLSHxe9O81DdTQHvXKSj3GY31TpjnD27xUFXmp2htpHMWdZhDwGjG79js59sN2IhmatWtwOWuyzVqSVfspdv08NsmH0yoD55KYpNR6Eod8E5rsWUAY4RdxlAMqWA18k3Mpq3YlTmUYK8Dmg51g9hwgqPFLlxj+lWZgBPxAMs9NVX8p9owVmCjnCSAmC0pR4j10JMyVBicVIBdVfkt5/vvjw0v7//fPvp4/2DEo+8lMTQuwBkewdk4aTkNFLkQpgDgK8vWlrIuxIQVpsEDBJ5Ya730Y0TY1GCC5ArlUtKr7wOHMU0B+ORMVg9Npo5vHt3/aZ8sXXXv2y3221db931jZx2QVNE+EOPE34MwYUi/2Kj7vEss0VA+TOEC2BJVWShyH9ITnnOq5zdI1o6YZhbUK03ZrU8Hj22jB2Q9RPDrJVLHJ10X2b6+gJOejuH1+cx/ZnHQd/8/Cb/Sx2Q/6ezF9a6XoUrvRXCUl0l5cR5BskQWXNRqshNpBNe1c8Z0z3N5ZmZtp0sSCGB5VV9ti9aADa+TjlFjU2c+p6OyVfN51d5GjbpcenHIlvlPNpEMKk1hrZi42dr/rTPk7WLvJvXXLXO/y0I3f5bIc5d5OrXl9fbm5/SZ8lNPfwHOBv/L9wrY/zgQpj8jyleymkH+auVOQhxFdCPusW5N9nqshVp9OuSSnvYMB65kL2TrZMCVUcHjEmsK7ogRhPnVaQ+EUwVErKreg5dIXmbqN3/9h4MRaO5HYTYwicgT8Gm31F1Lpn9A1BLAwQUAAAACAAorFBdJ7x3Kk4CAACqBAAAEwAAAHBoYXNlM19yci9jb21tb24ucHnFU01v3CAQvftXcANSB22itIp25VOrnnNozmhsxmsSDAjGuxtV+e8V2GkiVe21FzTM13tvGOwcQyI2QZ6c7dunHHwbcpvAmzA3Ywozi0Alxuya+gA0NZvtlzm+MMjMxzcXhTRMTWNwZHkCUYrlvmFTt0GoPMHt5y9CNuxsaart1iwVInrBU89laTnuG8bGkFjPrGeWMAkHc29gP6qEYMT91c3u9q4esu05l/tJLdEAoehlwxLSkjyb1IQXY4+YSciVF1CY7aCL1orchv6pcCx2987nUE4VIaEnNT8bm8R6yd2PtGCLF5tJh+d6kw2jOXa1pOjSeRlHe6md1Gp/4ormyNdMdU6WUBNeSBQiyixzzCL0Ty36vCTUkAdru+/gMrbWG/TU3bbgXDhrD34NyBb9EIz1x44vNF7fc3kIWSWMDgYUNMe2StneA074PwQf6k6oCl8E0hz/SRPRaHBOFKPQXJdRlevqO/io/nSuKDP4BZx+9zfMjutWqmExoGzWcALroHco5P5D5EPpO/7GKSxpQF12GLOopEKgdXZaj9ah1nIb3e/V+xlVQgdkT6gpiFIhFWQdQ7YXIff1f8i647HseA6J0NQ8lY4u9IJfcSkL/1hoFxghGXjDRNxm3HVcxRfOSg/lYcau44/eDsHgNyC4vrlTO7VTdKHyq7xhXOv4MsAwodac+UAFORbqlF9XsRMkc4ZUpvOuhddB8W1eWp8wZRu81i0/xuXNX+d4RNIGT3ZAXRiJXZXw1ydg6DIy/vXhkbe8xN+abRC15rX5BVBLAwQUAAAACAB1rFBdlvnqlYkLAADoGwAAEQAAAHBoYXNlM19yci9kYXRhLnB5rRlrb9s48rt/BYsFjlIty3G7t3eQT4vrNi0uQLYN0ux9WMMQaGlks5FIHUklToL898MMKT+SbLsH3JfEJOfFeQ8l204bx1bSQumSjbCbRq6Sr1ar5F52tWxgVBvdsk44PGHSw18It0kuegMX2sotrkbhRPVtd8eEZarzmGmp21arAdNuRCKcbmVZIJfR6PzdLx/Ov+QL/lGUjif8zNoeeMLfmXXfgnIX4KSTWoE52LwE22lVgUKML0643iHOqbTW710YKAHPL6GRUP3WaXW4+0n/wcHnGzCmb6DiCb/sG6nWv9yd61sw73VvkPClcFJ/rq82cAqltJLQB8gLA8h/gP2kFfDlaFRBzUT5n14aKCrhRFTrpgKT9F2jRQVVcS+7HIGTSt8q3Ms/isZCnI2YB81RxQEtHjFZs0PcbMTYrXQbFkyW/i67j7KB6BAoRqPcIyirtWFS1ZpJxe5T/NVI6yJkxxjr8iPDRnieIlklWogJRNasS6UtxMrqpncQxUwbxtOUI8ku7YRxNjNCWmD/Fk0PH4zRJuK/KStqYKgD9vvZBd9RIx7SFpU0UZyVWjmpetjzQtZI+oG/Py+cEVKl6D08wXUFN8Pq7Or49Ozq8LQRK2iKG12K1QG6A+sO8XfLR68PhrfJ+RVYx4+lib6JHjNoLDB+hQJxT6oC63JvximSnXpic9xHrYFyaXuNWvALm1+ZHhLYSusKfU0rrzRSYn6fGhAVmShsy5qYpIRiB5vu9xG+WN05sFH8KkciL9jpVNY1IH9GZKRae5uhF2SMj60zEVKLx3zOegtMMAW37PTd1bvi8vPnq2BYUkBGbG+NdBD4IinvxIO7o5CULDb9ei3VuhYlFJt+l242Na6KARyhtSG7JKg+613DKzpb/H99ZJl40we6f2Tt5WMqHbSDxlG+wU1IxBBbwm2O7I9ng5sr7Qjgqe12mbN3svFbVvemhPyJXiL+Yds12mCKUpNzsZq+/zL5299nk/eisw6TUTLEcV7zi42wULwtLi+nDyjL4/QBTx55YqDThbvrIOdoKwtuMCiJ92f9dO4lTkvd3b2JvMgJUohHzIDrjQrZzWfISpeU222EMEmlWyFVQvaACnWhV19z1HWKd7URJSci553awdZFoEpdSbXOee/qyd8nVq557J1NWqmsE6qESK++JpUsHWlY1oyuSblLr74yoaonwAsPsEwwT8aZ0bc2P9geoacjGVkdEeEo0rDjsRZ69dXDWwg7D4iWXcNd8vr1DQbh4xNpadPL61PKHsNzyAIaOh3u0jKwHbxyOfIyPtHC8YVGQa6XEnffoRdCxS4vKQ8wzNJCYl0me9q+cfliObcAKrfgohiLl2FG36IoyCJoG/38QAyjb8PlsIagCgmAUGjLa3G/+YJ8H7YdlCjc4ERMVlPEY7WEprJzJpVFECZK14uG4XWw/lNeI+++hrsc85rRtwuUYhnPsZaDKsHmtElyLOfkkbSVrsFFPmNYoiFrJINiohZeSqx918hSONgLenbK+Pga7hAfsVJRVVFYP1fWTiRvOdQPQoimiQ6gtol1Jib1b700ASt+QabLS0aaanvr2AqYNhUYqBgyWFhnlsPVQiiSa0dP5PJK2AvVgAp78ascF3sR8Fiou2g7mNR3gHtpA95Lsn4JVKYEw1ppW+HKDZLs1bXStwGbRPY+mYquA1VFB0HjUwvPQooJQbRX7WDSzP8/vDvFH7Zqj/sk5vn4JIapqdB1LUspmqFhe7mNm1OROcxo4WTqK9n0WSn6M4nO99w3oCptUsBMIBwUHeb6t0MlISGJbqK0aUUj76GQFWVJjNz9eUR/4/gVRbQ31Et2+RxuzAi+b4S5YxU1EdZLNLg7VLO3wcBoo8rmD48+T9iukeQOEafyzBNewQ1H61jHKVNXdkFQy9z2bRTtC0bQW7Rv0gguzz3yUSMWT2v+UD0WDwTyGIp5ldDyVcDwsVMNTR4Vex7HyWJJsVDZfLH1WWLvtjvpML+jy4CKHg71G20PQlJW9jGEhqxejMv359Ozq8M8YVmpm0ZWwETtwLDBzVjqto4NnKj4o24Fwtr84Tp7MAeyYuvPbkKxSG685EOVeCQfwOj02Aux/Ev4tfLoIlmRVo6tFCf7tVdgEh2ZL/6m15DeLNM3YBrRkUgH194nb2pCV1BrA4zYSbX2brTgtVRrMJ2RyvFl/tClBhrh5A0UTg8hlwpbdDjYRHFmNyLqvEGIo6XqFiBTs270KuKvQ+ChYkK0V3Y0KhthLbvS16DkPZhsxDD2i0Iq6YoistDUvtFBr8VVSo3fvmOZP+lldjB/HOEU3WgdLOpUeGq0uONxnvPSYpunb9UEW+zJqoPJzYz7ThTDj86H8Ne3qlh11Hb+wD4buZZKNKzTjSzvsJu37DclS13B7Ed2Pv11+qlidGGwKbuQisFWlK65Y6I02lp2cec2WrEbMDgN2xTpHorZe2JFAODxq5zPfkxP0hP+nU5DD7IFeQYevht1YtWA12lRYEdSFHHoTafcJ8BpQDwVTkw8T4wVGsaswyEVOxZQGM/LeSn8GssF6h3T/NDLN1JRR0VMv2WklHwZwXcNvG9BctzzpxGf83iOguVSucifL06WyeyneI49eFIKl4ftWfYWE4pvBkQLKYqL037EE/ZRGut+5nE2yIxE50fzc40geTh/kci5IBo+SxKBpxpAtFK4xckSdcDPf+VYcVHInH+qeOZ1OVRa4hiTWoctpBqTgnc7wpEVfSjtHM5rrFcSQ31wmZ2/0OE/rRNOli24jcZREKdqEg/W2txF5W5syv2rVur/FUauNy7yoibaVFEZx5OZhw2xjQIu5JL675/zE+p0POQ/crwOnvla8t5P9CGU0uDjNJPv7kJehoEt267JB9DPt+qXiw8U9z4XDNPALljdkFfsELG7TINvOP/6uFseczg4SJEQBUWEPS3xwTwJNSOPxUayqRN04TgLl98RSgMInaYSEx4idgYwugrYirZrcI7XpU2cvk5asS1IaJu/+etPCbofbF2xb59naJP9iOBMr9BeVX7ia3+ly5D0aUag96lkQMcTUH0LRjhApkMn7g3thFmDy52+HuQeEMm/QiH2UPHPe1Enb7KdHOOc/OAHdgoOTCsVPnqUbAOiGjshG1ZiaRJr8LlAq/W+pZ4zHJeNY4Ga1CphCm6wQLfShWYn/Z4s5Icqjw434+n0zXy4H/1bZGo5Dj8nR7ATFWeUJ8LM4IGygL3wOWTVV7g6wjuQCEEaqI+UyRlPv2qpDhW/aMU2Oknk5Jml40wu45jSxdMjHzgLkoNC8btc5HiWyfFs/IzUn2HR5K1UEY0hULs48VefTt/Ec7M7IjmGs0kTz1/GmuCbKwuNR44ni0mTUZpoBp5z6slyorjIjDcF1pXZcuwRg93GBDhevFnOW2Gv88XJ8nU0GyNPDxfH48Vs+frALmOCwQ3CjcczkufpfFPpEmea0jd7h2NOuRgWy4RLVcGWZzLhsrI8k5VNOErCM/ybkCNy7JKokczkFK09S5D9YfBNZnEYlHjmB4OUKHug0OIvF3JJtjraZNKSj2J58fqbzJ7NU8kD3xmWZ2Qt2o8T7tWyt3uxC2S+j2m81ODmeLXhd8Kf+Q3Pnm09+px3IxqsRFB0BvCNAhucCH8nmKswAYWRCffiV/lDddBp0/iAcI/f6n3PTndjLPY1R4jhwQQvjwwWgfoyDA7VzhjfnamP0V+Q50zRXXd5jZW6V87PoDz22oAtJjr/PHekhO+oaf60+/2fnhFHzLVdTijYsRS2r2u5jTh9ZZCCZv0Xv364tkv4LR8+ZKW/n10Upx8+nr+7+nAavobc+4dprJF8L3uYCak/r/q2C3cBZXsDhbCllOEDzTdY77+3yJrd0zO//8ryKl8847VEA96nKMm97KL4MEK+YazwDcW1OO10jSghKHgo6TjjUI/xX1BLAwQUAAAACAAorFBdxdAJ4goKAACSHAAAEwAAAHBoYXNlM19yci9tb2RlbHMucHm1WN1z2zYSf9dfgfPNDcGEhiP50utIw4dc2/Q6k7QZ170Xj4YDkUsJFQgwAGhZzeR/v1kApChbduJO74UfwH5hd7HYH87Ozn7ZqfNGVyCJBeVAlUBKya0VtQBjCVcVqY1uzm1puCs35J2+epPh8/0bP/nTm0s2mVxBDQaZ7Zz8pyPgCJds9mo2JRQZ0gX5t9hwtb/4wB3Ii/e6ErNXs9d+9v2bdDF5J0Zcs4y8hd35rxvtyAdueAMOzPkPdS1KAcqRt0LB+XWnhFoT+tOby5RNzs7OJqJptXHkd6tV1nC3yezeTtB40nK3kWJFIsUH7jY9tdOm3AQq/9nTKDUaZEr143WnSie04pJwS96OiTonpGXlBsptq4VyPcthJFCzUjeNHiTaDZ9c/fLLdY5m0aKohYSiSFnLDSCLNqQlQhGKRBfJLahKmyQb/12seLldaQU2SecTImpinaFtSpR2yGr3lqEP5v0HE8qCcfRVFgjTycRHnbzTuyuutlQp9l5XnQSUV0FNikIo4YqCWpB1tuIWsgbcRleZ4Wqbf5tx2W54Pv0GGYjtWjA0ZQNXukA+hnw5PsJvkJCHVxiyJZeQe2EXKBmF4XildypXir0TCrihKIMJVdTAXWfAeiuyleA2f8ulhbTn69oRVyC6x4ruiobkidSm4QkBaYF4Qt25g5IT8iujW905VPJ9+KTs1Wucfih2rhRDf7A/wGgbXOkXxnYg1huXLnqCLReNUOuiU6LWpomkXRsJM55jgjP70Tj6OkVtaPH8afaRpvsCFqcsG9SlIQVqbXbcVCED7nycu7biDvJIflATv6JD7tI0fXGI7oQQA64zKngQ3UzvXgZRqU/eUW4ch2TEkEaOPnN/enP5VVmreAPPyVGh2s4VIS2RN8+T3SwJc3VnoRpGP25vkwkhKj+VYfdlPZJiIW8OwtFQHFH/uJwbLiyQ/3LZwQ/GaEOTt0hCPm5vsYi0QBphG6zSCWZEkBINV2wopDRUK6wWdPZCXVxcHhLoK1hU+ng6nFjn/GGoR6nAnKZ3rHL7FrwR+3wcYQyStyWoL7mjY0v2zK/65ny6vLi4zCq4FSXkexY+0uygJk2PvTpKpnsZuX8x2LWPdk0muFpe8dZRf1KOC18oe7HoDilbd1Im/cpvlhPi+ZiBj50wYIu14VVBQyFZYD22OVJJvcs/hTzKkp3Gx0d8bPHhx6b4mOHjMvk8IYKP6A9UM5xDo3dcbtHmTkLWGqjFXZ74EwKDRzBvs3IjZIVnhBTWr6+TwHCmKvyUAUVTz4IrFFYo67gqgfrZbKitqe8GkNGfVYIfOUTwy7iDpd4FaYf5/pSiuNV5ksUdn3mu9ETa/6a2Su9UiAnH0zgkPLHguHOmX7JfINaFaCvu/EfM6s+9QPkwwDFMjLctqIoGZ770ElGz3zze20HAaP5lwtC6PhQgU58ruGYv8sT6ftaEl67jEvOmk2DDSqFCQTGrPG9ITal5VfQdAF11qpKQ1bwRco+ubvkeKeImwk/qO41AmF4kh/6EtS7JGt4WUpfer3lStl2ShWPAFlrJ/XD+iZoEJXmeVLPZLPF5hQ2OhDWXRcVCXxnbnO+/06oW62xo476HUldgJoSU9TqvROloNPUmgVtRYTuZLG8SL6UoPXeyTBdNfl8EjbLpixdlvU7TxU5UbpOX9fom8Z/JEgtclJmf0LKoxBqsy/uRm6TijidLtgZHE6e3oMQfYAq74bPX32AYfMT7BZf2X99OWamVM7x0tl/ye7Q8mHZM6ZdkGe8J36HDfvxwfZJqaFt/UH61cdG9O48GvSCv9qSoshd1BWVnsL2MUob/IwEPPdaA48EziyFoB5/dj9Q9pz7mzHG4qsIL8QEbJxh/lcybvPcTHXm2D7o/wsYsqymyPOYfesqbj8oqvawTXqL3PHkkwcITxSvIxlQSNQme8iU0fP4ttxt+b58OnmMIcU5WxuuehFSirhHABfxjwBkuFFQjMIKqG18PCuu4g+J4C8Y4IEAQpcuvTYeb3lP2+RAC6oeS7NPndIHbPc4NharJfHizT8fpMS/rdZYEjBP0J3P/yh5myPzLvjiIepTluMyln/vG8erquwHwfrmDDFU2lte4NC8HbD69/EJjGZjz/iM2kSHJwquHFRvgFbZgv8LHDpQTXKJp7/gezM/aNNQrfjlLszHumL72/xHsRJLeur5tM9AaQKzvC7wN6xKV9aY3ff8VDBx3o8elvg8u6xtXiiJuXi1PseDmxVPyLm+Y09sCmhVUlVDrwIVT2I+spC632Ag0zH/a0Cfc5YfIUcmbVcXJbbaS29xTzVdyS29RdXaXdRYKA4B1WLl4Ug32+C2Adwa4zRpW1muGjRgeJMVBBRKE3hyl07u4qCGdmUL/++a0gUabfXbLpajyhoGvKLik7NClFlJsw5jvJuMpvNJa+iLxwFdYtZ7nqypUsGc4Kxsb/v/z3AM1R26MZhcHd+JuEVxGFzmuNrTxqFRwWbRG/w7+5oUGsawBrug0TVNWS80dHYuI75uftYIlg7uWq4oGw1WxNl0hcSfZ7HyanU9Tf3KLdac766XshNvEOx3eOV1y62gAFYWPoahsBBkM/zNQfCWhil6b32JBtlmRN2xtOnoyjr3NWbQU1f6dfNhf+zuoH69+I7pzbeeIsOTth8uZD/r1P7H+roQUbr8g3DmsDFoRAw0XypLeWDYgc29EabS1BXdO0f7HOz3YmYb4xGDl4YXm3OWBoMfZo9gF22LoXjYIqnxg4kcf05NIEbdCw+02a7UVaH7AI7GMhPpzv0bFzN/lyIjYrCePEG3Rao0BGMZf7G7m2Tzz4U+Z7Ro6TS924SPbArSVaMKRxkrJm7ZoBObShJBW27y3DDUFyVEPM+AhJz2fZp56DG6xZNMRUg2cuMxset5qm2ZTvGdDhzR8C6HROW7Xn7hSG06ecOQMrdlTzf9k6OBukoggiogokmUeEO0g9yHmibA1PzogT56AI00sZAv9lMQGZx4JsXfc6CqZR0UJakrmXl/iFSZz/8omhOBpHq4ebDLHqLVMdQ1ImpLhPjRg6gMhTdMs8ZUKd2PxpyRgxWuPcXr6+QC6AvzvlxpiacA6baCwIKF0UFFEZv1lRFJ2FfeI++zs7CpQErcBUuqmleD8JYRnI1dXobny1Ud3jigArBeeXBuxFnjnHLocgnfEDO+8T6E7b8DXobjFU6Bo0ve/hyyKMQ3d/2ON/18JDYee6U9hvuMmPrYhz8ZhX8B1g4lPoZNHMM5DxPLXQsCD+/4MDPoqoxEa/RVgczD1GSjrUQOfRF7jLRfy8IDF7hXFUYqH6rUcDfkCNh4IVWz5nMK5CDXomSisOoWzvCCnY6eSMrjlkqb3qpUDZbWx1OhdrFFYnKKIeHoFGnpj9O4mEZVNlsu+nvVXqycI8Wj+Osr+eH1InU7+B1BLAwQUAAAACAB1rFBdYWWcUj4YAADHRwAAEgAAAHBoYXNlM19yci90cmFpbi5webU8a3PbOJLf9SuYbG2BtGHacmZSW9Ry63J5bKUuM0l5PLt1q1OxIBKSsKYILgDKdlz+71fdAPiQKGWyN/clofBoAP3uRsMvX768/eFCsxUPNFeClcHNTWAUE5Wo1jSQ99XFVha8DAqhjShLZoSsAlYVAX9guQkU101pAqbyjdhxHb98+XIitrVUJshl/UjXOf2nlhXdMrOhtcjvSk4Vqwq5pUZsOf0q6pUo+WSl5DaomdmUYhk4AF+Y2XhgVbOtHwOmg6r2TUaqfGMn4mdcVX7mqqly2CgrYcoHO0jflZypKt5yo0Su/dha8VxoIatM8ZyVZbbSuVQ8000N/TSX1arB/i0zSjxYYHEut1vZLsiM3Io8w6NqtuNUc15krCyp3jCqZaNynm2Y3nBNN0wV90y5M8cOdZmo6sa0u1Jcy3LHs2VTFR49ccEM8wNu5R2vxFeuaClZkcnVSuSClbRWvGaKZ/yBbeuSa/rpzX++//QL5Q8wzQFCkrZrbdkdz7CJGl5pqfRkUvBVwBojc6ZNWPCdyHmUKG4aVTls7/Vm5rHmqf2O4ZsW2GIHr0rJzPQ15RVblrzoj0tTkjcFI9Fk8h+OjjJbK1aEEe6ilGthdLaSKrR7VPJeUwuA1kquFdc6/cBKzaNkEuCYmO9YGUazHSsbrtP5Yrbka1alwHIx/BNGk2AlVSAAWiCqgFfNlitmeAjgAVBwL8zmEAcIMsVVwjOHLpjjdhRF9qywQCBWQSUdo8ZCr0QlDA8RQhSzsgyjRDGhefABpohq/UWKyrxXSqqQ/CwrOyEQ1YorXuWcAFB7ppjVNa8KC2x+tYjzugmjGMUkjNziHjsor6E4n0Z//PHqKk2vklqJyoQr8tGDDp7E+fT58qnklcXA8yzgJas1L4KnsIe3C8RkdPn6Komnq+etqAhdlY3epLeq4dEkcExS1bE2LL+zO9QR7Md+BrzUHPr5tjaPYXhFYVHLplFEq9pi8NV1ZNkQxRE3RR2sZBIY1ZhNOldzUrIlL8kCyamAlDByMasVL1KPK7XesodwGs2sukqfni35C7llooJJIWFlSSh5+4lQ8vGWIAeIqhA58I/wzDLCKnAwCydNEUgAG5kT20YWaWq/Fh1DOLiO+H+DTTqi/yS0FtU6KPjOAUWas7SqY6YUewzx4NHcgVjMlimctP09CTqFRq1Co6spdcos/YayCxldUkSoTkuhTahYtebh9FUU0a9cyawQO5yeXsG2LDbn7nzpE9myXMlsNSWJFYLVNN5yVoVRRF1fuwE/pG3YH2k36IfZX+2YSRAERPPKAO9qkgAHORRElNRcZZYtkqeKbXnyRPp7mosFDDq6E9s/tr7tccgiCciQ+wE9z45LYMkhozjmfrb73rcoJNlvOUGHKDYSm6LnVtYsHay0OP0fAkixptZ+ULAcmhvgaiWlScG0uiFzopoqg1ayiGbwf7y9K4QKAUxlNMo15Q9Cm0zeeSlfiWrNFaqR9IkAeJK4Reak10kWlBhvqkiiNyy0W7rsmmMwmwQJ/5Bhq7Yogc312hYUUGf4g8l6pPcDD7sWz5MgZ/mGp3CqbsEiru9KMgFpxO4YD6fDTuvbZlnzKiRqSSLwIlaJXP4ztS5MDDY3XDk1K5f/3Dvzi7T3c0TMnfG2zhQuFhRiteJKz4J8I6XmAQsqfh/c/PpzdvP58y1qAUdtXA5UD1lMAgMk8Yg8itoZDE+fnmfaMKNb9afrUoA2CkKCLh+hpOA7Qonh2lgVCBPnOG5BcbL7ke77GaGnvhts5B0dpeEJesEhrWXyGOIFoQhwsPjQ4JhtnVqCAe0y3axW4iEkcc2UEawE1CFRzbZ2JL1vSerIWTTbOnwa0DDp/aAW3Qn8+0yB7ABL8bpkOQ9x7WjWcwHDAb8hlS0h6Kk18IQkwf+eo4k3FkgBR58F2JauEYi1GGGvn6V14S/BjHT47YwzTLfqAsdlsjqiMKyzZTgcUWXWFUt/lhW6WiA/IxT+8/Q1bPMEnf981R/A8rzZNjaygNn9PoDLa5lv9H6Pj1GyjWwU9KZXI5j4WO1YKYo2ogmWTbHmBgChvx0UYssr0LwWPf8PutFC8nqKEjs10xs2VIf5hud3NTiAcW1AG7Z[REDACTED_CREDENTIAL_SUPPLY_OWN_TOKEN]...')
assert hashlib.sha256(source_bytes).hexdigest()=='2e32f2e722dbf031e98bc89af3a8bcae971b899b4e36b5a0a6bbb8818e5b59b1'
CODE_ROOT=pathlib.Path('/content/phase3_code' if pathlib.Path('/content').exists() else '/kaggle/working/phase3_code')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
    for item in z.infolist():
        p=pathlib.PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts
        destination=CODE_ROOT/item.filename
        destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(z.read(item))
sys.path.insert(0,str(CODE_ROOT))
print('Source SHA-256:','2e32f2e722dbf031e98bc89af3a8bcae971b899b4e36b5a0a6bbb8818e5b59b1')


In [ ]:
# EDIT THESE PATHS if you use Kaggle or a different Drive folder.
from pathlib import Path
IS_COLAB=Path('/content').exists()
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST=Path('/content/drive/MyDrive/CS781_Phase3')
    INPUT_DIR=PERSIST/'inputs'
    WORK_ROOT=Path('/content/phase3_work')
else:
    INPUT_DIR=Path('/kaggle/input/cs781-phase3-inputs') # edit to your uploaded Kaggle dataset
    PERSIST=Path('/kaggle/working/CS781_Phase3')
    WORK_ROOT=Path('/kaggle/working/phase3_work')
WORK_ROOT.mkdir(parents=True,exist_ok=True)
RUN_ROOT=PERSIST/'runs'/'02_B1_LoRA'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
required=['Phase3_B1_Input.zip']
missing=[str(INPUT_DIR/n) for n in required if not (INPUT_DIR/n).exists()]
assert not missing, 'Upload these exact files first: '+str(missing)
print('Persistent run:',RUN_ROOT)


In [ ]:
# Official data: upload ZIP OR authenticated download. Existing extracted files are reused.
from phase3_rr.data import acquire_data,load_official
DATA_ROOT=WORK_ROOT/'Phase_3_RR'
DATA_ZIP=INPUT_DIR/'Phase_3_RR.zip'
LOGIN_HUGGINGFACE=True # set False if your token is already cached
expected_files=['Train/'+n for n in ['CL_train.json','CL_dev.json','IT_train.json','IT_dev.json','label_vocab.json']]+['Test/CL_test.json','Test/IT_test.json']
if not all((DATA_ROOT/n).exists() for n in expected_files):
    if DATA_ZIP.exists():
        acquire_data(DATA_ROOT,uploaded_zip=DATA_ZIP)
    else:
        if LOGIN_HUGGINGFACE:
            from huggingface_hub import notebook_login
            notebook_login()
        acquire_data(DATA_ROOT,download=True)
dataset=load_official(DATA_ROOT)
for split in ['train','dev','test']:
    docs=dataset[split]
    print(split,len(docs),'documents',sum(len(d['text']) for d in docs),'sentences')


In [ ]:
# Runtime estimates come from ACTUAL GPU calibration, not the initial planning estimate.
# Change RUN_ROOT to a new name before changing any setting of a previously started run.
CONFIG=dict(name='02_B1_LoRA',family='b1',method='lora',bundle=str(INPUT_DIR/'Phase3_B1_Input.zip'),
    data_root=str(DATA_ROOT),run_root=str(RUN_ROOT),work_root=str(WORK_ROOT),seed=781,
    rank=8,alpha=16,max_tokens=256,context_sentences=1,accumulation=8,
    backbone_lr=0.0002,head_lr=0.0003,max_epochs=8,patience=3,
    training_hours=3.0,save_every=50,temperature=2.0,distill_weight=0.5)

print(CONFIG)


In [ ]:
from phase3_rr.train import run,package_records
RESULT=run(CONFIG)
print('HELD-OUT macro F1:',RESULT['metrics']['all']['macro_f1'])
print('CL:',RESULT['metrics']['CL']['macro_f1'],'IT:',RESULT['metrics']['IT']['macro_f1'])
print('CODABENCH ZIP:',RESULT['submission'])
print('SHA-256:',RESULT['sha256'])
print('No private leaderboard score is known until you submit.')


In [ ]:
# Download the prediction ZIP, NOT the source/model archive, to submit.
from pathlib import Path
submission=Path(RESULT['submission'])
assert submission.exists()
if IS_COLAB:
    from google.colab import files
    files.download(str(submission))
else:
    from IPython.display import FileLink,display
    display(FileLink(str(submission)))


In [ ]:
# Optional larger archive for records; not a Codabench upload.
records=package_records(RESULT)
print('Selected weights/source/tokenizer/config/run records:',records)
print('Full optimizer resume checkpoint remains separately at',RUN_ROOT/'latest.pt')


## Resume / troubleshooting

- After disconnect, rerun the cells with the same Drive paths and configuration. `latest.pt` restores optimizer/scaler/RNG and epoch/group position. `best.pt` is selected by dev macroF1.
- Preserve both files. Do not replace input weights, edit source or change hyperparameters under the same run directory; a fingerprint mismatch is deliberate.
- T4 does not need BF16 or FlashAttention installation. If memory is insufficient, use a NEW run name and reduce `max_tokens` to192 or128. Accumulation controls effective batch size, not activation memory per sentence.
- A Hugging Face401/403 means dataset access/login needs attention. Upload the original official dataset ZIP instead; do not substitute a different dataset.
- Unsupported data schema or colliding CL/IT IDs stops safely; inspect the official supplied evaluator rather than silently rewriting IDs.
- The upload ZIP must contain only root `prediction.json`, all official document IDs, and one exact official label per original sentence.
- Training code can run correctly without improving F1; inspect per-label/domain errors and compare methods on the SAME dev sets.
- Cite LoRA (Hu et al.2021), LoRMA (Bihany et al.2025), IA3 (Liu et al.2022) and distillation (Hinton et al.2015), plus the capstone/IL-TUR data. Declare coding-assistant use and actual GPU honestly.
